#148 Просмотры по типу монетизации за февраль

На каждый день за февраль 2026 года выведите количество просмотров отдельно по монетизации SVOD и AVOD.

In [ ]:
SELECT 
    cw.show_date::date AS show_date
    ,COUNT(
        CASE 
            WHEN c.paid_type = 'SVOD' THEN 1
        END 
    ) AS svod_views
    ,COUNT(
        CASE 
            WHEN c.paid_type = 'AVOD' THEN 1
        END 
    ) AS avod_views
FROM content_watch AS cw 
JOIN content AS c ON c.content_id = cw.content_id
WHERE EXTRACT(MONTH FROM cw.show_date) = 2 
    AND EXTRACT(YEAR FROM cw.show_date) = 2026
GROUP BY show_date::date

#136 Пользователи, смотревшие видео каждый день в январе

Дана таблица watched_content с информацией о просмотрах видео пользователями.

Выведите список пользователей (user_id), смотревших видео каждый день в январе 2026 года.

In [ ]:
SELECT 
    user_id
FROM watched_content
WHERE EXTRACT(MONTH FROM date) = 1 AND EXTRACT(YEAR FROM date) = 2026
GROUP BY user_id
HAVING COUNT(DISTINCT date) = 31


#135 Статистика просмотров видео по дням

Дана таблица watched_content с информацией о просмотрах видео пользователями. Каждая строка в таблице соответствует одному просмотру.

Посчитайте в одном запросе:
- количество просмотров  видео по дням
- размер аудитории  — уникальное количество пользователей по дням

In [ ]:
SELECT 
    date
    ,COUNT(*) AS views
    ,COUNT(DISTINCT user_id) AS audience
FROM watched_content
GROUP BY date
ORDER BY date ASC 

#117 Время доставки заказов

Напишите SQL-запрос, который вычисляет время доставки (в целых часах) для заказов, соответствующих следующим условиям:
- Заказ был принят 1 января 2024 года 
- Заказ не был отменен  или возвращен 
- Время доставки рассчитывается как разница между временем первого статуса Accepted и временем статуса Delivered для каждого подходящего заказа.
- Результат должен содержать только целое количество часов, без учета минут. 


In [ ]:
SELECT 
  o1.order_id
	,FLOOR(
		EXTRACT(EPOCH FROM (o2.status_datetime - o1.status_datetime)) / 3600
	) AS delivery_time_hours
FROM OrderStatus o1
JOIN OrderStatus o2 ON o1.order_id = o2.order_id
WHERE o1.status = 'Accepted'
	AND o1.status_datetime::date = '2024-01-01'
	AND o2.status = 'Delivered'
	AND o1.order_id NOT IN (
		SELECT DISTINCT order_id
		FROM OrderStatus
		WHERE status IN ('Canceled', 'Returned')
	)


#81 Гистограмма твитов за 2022 год

Вам дана таблица данных твитов Twitter. Напишите запрос, чтобы получить гистограмму твитов, опубликованных каждым пользователем в 2022 году. Выведите количество твитов на пользователя как группу и количество пользователей Twitter, которые попадают в эту группу.

In [ ]:
WITH tweets_by_user AS (
    SELECT 
        user_id
        ,COUNT(tweet_id) AS tweet_bucket
    FROM Tweets
    WHERE EXTRACT(YEAR FROM tweet_date) = 2022
    GROUP BY user_id
)

SELECT 
    tweet_bucket
    ,COUNT(*) AS users_num
FROM tweets_by_user
GROUP BY tweet_bucket

#84 Процент подтверждения регистрации

Новые пользователи TikTok регистрируются, используя свои адреса электронной почты. Они подтверждают свою регистрацию, переходя по ссылке в текстовом email сообщении об активации своих учетных записей. Пользователи могут получать несколько email сообщений для подтверждения учетной записи, пока они не подтвердят свою новую учетную запись.

Старшему аналитику интересно узнать процент подтверждения регистрации для конкретного списка пользователей из таблицы «emails». Напишите запрос, чтобы узнать процент подтверждения. 

Допущения:
- Аналитика интересует процент подтверждений конкретных пользователей из таблицы «emails», которая может включать не всех пользователей, которых потенциально можно найти в таблице «texts».
- Например, user_id=1 из таблицы «emails» может отсутствовать в таблице «texts» и наоборот.

«Confirmed» в «signup_action» означает, что пользователь подтвердил свою учетную запись и успешно завершил процесс регистрации.


In [ ]:
SELECT 
    ROUND(
	    COUNT(texts.email_id) / COUNT(DISTINCT emails.email_id)::numeric, 3
	) AS activation_rate
FROM emails
LEFT JOIN texts ON emails.email_id = texts.email_id AND texts.signup_action = 'Confirmed'